In [2]:
from dataclasses import dataclass, field
from typing import List, Dict, Optional
import pandas as pd
import numpy as np
import rasterio
import geopandas as gpd
from datetime import datetime
import matplotlib.pyplot as plt
import sys
sys.path.append('/Users/michaelfoley/Library/CloudStorage/GoogleDrive-mfoley@g.harvard.edu/My Drive/Subnational_Yield_Database/scripts/global/boundaries_processing')
import relationships
import seaborn as sns

In [3]:
from datetime import datetime

current_date = datetime.now().strftime('%m%d%y')
print(current_date)

010826


# Assess what states and districts we have as of the 1991 census

In [4]:
india_relationship = pd.read_csv('../shapefiles/relationshiptable_IN.csv')
india_changes = pd.read_excel('../shapefiles/IN_Relationships_units.xlsx')

In [5]:
india_gdf = gpd.read_file('../shapefiles/INDIAHIST_DISTRICT91/INDIAHIST_DISTRICT91.shp')

In [6]:
#Step 1: Mark the unique IDs from the 1991 boundaries (Taken from https://hgl.harvard.edu/catalog/harvard-indiahist-district91)
unique_ids = india_gdf['DIST91_ID'].unique()

In [7]:
# Take unique IDs and make a selection of those district names and state names
districts_1991 = india_gdf.drop_duplicates(subset=['DIST91_ID'])
districts_1991 = districts_1991[['DIST91_ID', 'NAME', 'STATE_UT']]
districts_1991 = districts_1991.rename(columns={'NAME': 'District', 'STATE_UT': 'State'})
districts_1991 = districts_1991.reset_index(drop=True)

print(districts_1991)

     DIST91_ID            District                 State
0       9999.0  DATA NOT AVAILABLE       JAMMU_&_KASHMIR
1        453.0              LADAKH       JAMMU_&_KASHMIR
2        454.0              KARGIL       JAMMU_&_KASHMIR
3        465.0            BARAMULA       JAMMU_&_KASHMIR
4        466.0             KUPWARA       JAMMU_&_KASHMIR
..         ...                 ...                   ...
463      355.0       CHIDAMBARANAR            TAMIL_NADU
464      442.0            NICOBARS  ANDAMAN_&_NICOBAR_IS
465      181.0              KOLLAM                KERALA
466      182.0  THIRUVANANTHAPURAM                KERALA
467      357.0       KANNIYAKUMARI            TAMIL_NADU

[468 rows x 3 columns]


In [8]:
districts_1991['District'] = districts_1991['District'].str.title()
districts_1991['State'] = districts_1991['State'].str.title()
districts_1991['State'] = districts_1991['State'].replace('Karnatak', 'Karnataka', regex=True)
districts_1991.replace('_', ' ', inplace=True, regex=True)
districts_1991.replace(' Is', ' Islands', inplace=True, regex=True)
districts_1991.replace('&', 'And', inplace=True, regex=True)


In [ ]:
districts_1991.to_csv('ml_infomap_1991_districts.csv', index=False)
print(districts_1991)

     DIST91_ID            District                        State
0       9999.0  Data Not Available            Jammu And Kashmir
1        453.0              Ladakh            Jammu And Kashmir
2        454.0              Kargil            Jammu And Kashmir
3        465.0            Baramula            Jammu And Kashmir
4        466.0             Kupwara            Jammu And Kashmir
..         ...                 ...                          ...
463      355.0       Chidambaranar                   Tamil Nadu
464      442.0            Nicobars  Andaman And Nicobar Islands
465      181.0              Kollam                       Kerala
466      182.0  Thiruvananthapuram                       Kerala
467      357.0       Kanniyakumari                   Tamil Nadu

[468 rows x 3 columns]


# Load in Name Files from D-15 Tables 1991 Census (https://censusindia.gov.in/census.website/data/census-tables# - Age, Sex, and Marital Status C-1)

Cross-check with IndiaStateStories (https://www.indiastatestory.in/districtdashboards) records and wikipedia

In [10]:
import glob
name_files = sorted(glob.glob('../shapefiles/district_name_files_1991/*.xlsx'))

In [11]:
census_names = pd.DataFrame()

for file in name_files:
    df = pd.read_excel(file, header=9)
    df.rename(columns={1: 'File', 2: 'State_Num', 3: 'District_Num', 
                       4: 'State-District'}, inplace=True)
    df = df[['State_Num', 'District_Num', 'State-District']]

    state = [dist.removeprefix('State-') for dist in df['State-District'].unique() if 'State' in str(dist)]
    unique_districts = [dist.removeprefix('District-') for dist in df['State-District'].unique() if 'District' in str(dist)]
    print(f"Processing file: {file} with state: {state} and districts: {unique_districts}")

    if len(state) > 1:
        print(f"Warning: Multiple state names found in file {file}")

    state_df = pd.DataFrame({
        'State': [state[0]] * len(unique_districts),
        'District': unique_districts
    })
    
    # Simply concatenate - no need to extract unique values first
    census_names = pd.concat([census_names, state_df], ignore_index=True)

# After the loop, drop duplicates if needed
census_names = census_names.drop_duplicates(subset=['State', 'District'])

Processing file: ../shapefiles/district_name_files_1991/1991-C01T-0200.xlsx with state: ['Andhra Pradesh'] and districts: ['Srikakulam', 'Vizianagaram', 'Visakhapatnam', 'East Godavari', 'West Godavari', 'Krishna', 'Guntur', 'Prakasam', 'Nellore', 'Chittoor', 'Cuddapah', 'Anantapur', 'Kurnool', 'Mahbubnagar', 'Rangareddi', 'Hyderabad', 'Medak', 'Nizamabad', 'Adilabad', 'Karimnagar', 'Warangal', 'Khammam', 'Nalgonda']
Processing file: ../shapefiles/district_name_files_1991/1991-C01T-0300.xlsx with state: ['Arunachal Pradesh'] and districts: ['Tawang', 'West Kameng', 'East Kameng', 'Lower Subansiri', 'Upper Subansiri', 'West Siang', 'East Siang', 'Dibang Valley', 'Lohit', 'Changlang', 'Tirap']
Processing file: ../shapefiles/district_name_files_1991/1991-C01T-0400.xlsx with state: ['Assam'] and districts: ['Dhubri', 'Kokrajhar', 'Bongaigaon', 'Goalpara', 'Barpeta', 'Nalbari', 'Kamrup', 'Darrang', 'Sonitpur', 'Lakhimpur', 'Dhemaji', 'Marigaon', 'Nagaon', 'Golaghat', 'Jorhat', 'Sibsagar', '

In [12]:
census_names.sort_values(by=['State', 'District'], inplace=True)
print(census_names)

                           State                    District
433  Andaman And Nicobar Islands                    Andamans
434  Andaman And Nicobar Islands                    Nicobars
18                Andhra Pradesh                    Adilabad
11                Andhra Pradesh                   Anantapur
9                 Andhra Pradesh                    Chittoor
..                           ...                         ...
422                  West Bengal                       Nadia
423                  West Bengal  North Twenty Four Parganas
430                  West Bengal                    Puruliya
424                  West Bengal  South Twenty Four Parganas
419                  West Bengal               West Dinajpur

[442 rows x 2 columns]


# Check against shapefile district names

In [13]:
unique_to_census = census_names.merge(districts_1991[['State', 'District']],
                                        on=['State', 'District'],
                                        how='left', indicator=True).query('_merge == "left_only"').drop('_merge', axis=1)

unique_to_shapefile = districts_1991.merge(census_names[['State', 'District']],
                                        on=['State', 'District'],
                                        how='left', indicator=True).query('_merge == "left_only"').drop('_merge', axis=1)

In [14]:
print(unique_to_census)

                State                     District
48              Assam                Karbi Anglong
89              Bihar              Purba Champaran
102    Delhi District               Delhi District
103    Delhi District         State-Delhi District
107           Gujarat                 Banas Kantha
112           Gujarat                     Junagadh
116           Gujarat                 Panch Mahals
118           Gujarat                 Sabar Kantha
120           Gujarat                    The Dangs
126           Haryana                        Hisar
158         Karnataka             Dakshina Kannada
169         Karnataka               Uttara Kannada
182            Kerala                   Trivandrum
190    Madhya Pradesh                   Chhatarpur
265         Meghalaya                Jaintia Hills
291  Pondicherry U.T.                     Karaikal
292  Pondicherry U.T.                         Mahe
293  Pondicherry U.T.             Pondicherry U.T.
294  Pondicherry U.T.          

In [15]:
print(unique_to_shapefile)

     DIST91_ID              District              State
0       9999.0    Data Not Available  Jammu And Kashmir
1        453.0                Ladakh  Jammu And Kashmir
2        454.0                Kargil  Jammu And Kashmir
3        465.0              Baramula  Jammu And Kashmir
4        466.0               Kupwara  Jammu And Kashmir
..         ...                   ...                ...
449      450.0              Karaikal        Pondicherry
450      347.0         Dindigul Anna         Tamil Nadu
453      351.0  Pasumpon-Thevar-Thir         Tamil Nadu
462      356.0   Tirunelveli-Kattabo         Tamil Nadu
466      182.0    Thiruvananthapuram             Kerala

[63 rows x 3 columns]


# There are some that are still missing. The following changes will be made:
- add Daman and Diu added as two separate districts
- Chandigarh is added (seems to be missing from census files)
- Make Delhi only one district
- add a few to Gujarat (Ahmadabad, Vadodara, Surat)
- add Jammu and Kashmir (Ladakh, Kargil, Punch, Kupwara, Rajauri,
Baramula, Kathua, Badgam, Srinagar, Pulwama, Anantnag, Doda, Udhampur, Jammu)
- Kerala - change Trivandrum to Thiruvananthapuram (happened around 1991)
- add Lakshadweep
- Maharashtra - add Greater Bombay, Nagpur, Thane, Pune
- Tamil Nadu: check what's going on with Dindigul-Quaid-E-Milleth and Chengai-Anna, as the indiastatestory only has Dindigul Anna and Chengalpattu MGR. Turns out that Dindigal-Quaid-E-Milleth and Dingigul Anna are both valid names, so this merits an alias column. We will go with Dingidul Anna and Chengalpattu as those appears more common, but we will add Chengai-Anna and Dindigul-Quaid-E-Milleth to the alias column. Remove hyphen in North Arcot-Ambedkar
- Uttar Pradesh - change Kanpur (dehat) and Kanpur (nagar) to Kanpur Dehat and Kanpur Nagar and Uttarkashi to Uttar Kashi

In [16]:
# Manually add missing entries based on known discrepancies
missing_entries = pd.DataFrame({
    'State': ['Daman and Diu', 'Daman and Diu', 'Chandigarh', 'Gujurat', 'Gujurat', 'Gujurat'],
    'District': ['Daman', 'Diu', 'Chandigarh', 'Ahmadabad', 'Vadodara', 'Surat']
})
census_names = pd.concat([census_names, missing_entries], ignore_index=True)

In [17]:
# Manually add Jammu and Kashmir districts as per 1991 boundaries
jammu_and_kashmir = pd.DataFrame({
    'State': ['Jammu and Kashmir']*14,
    'District': ['Ladakh', 'Kargil', 'Punch', 'Kupwara', 
                 'Rajauri', 'Baramula', 'Kathua', 'Badgam',
                 'Srinagar', 'Pulwama', 'Anantnag', 'Doda', 'Udhampur',
                 'Jammu']
})

census_names = pd.concat([census_names, jammu_and_kashmir], ignore_index=True)

In [18]:
#Add Lakshadweep
lakshadweep = pd.DataFrame({
    'State': ['Lakshadweep'],
    'District': ['Lakshadweep']
})

census_names = pd.concat([census_names, lakshadweep], ignore_index=True)

In [19]:
# Manually add Maharashtra missing entries
maharashtra = pd.DataFrame({
    'State': ['Maharashtra']*4,
    'District': ['Greater Bombay', 'Nagpur', 'Pune', 'Thane']
})
census_names = pd.concat([census_names, maharashtra], ignore_index=True)

In [20]:
#Change Kerala spelling of Trivandrum to Thiruvananthapuram
census_names['District'] = census_names['District'].replace('Trivandrum', 'Thiruvananthapuram', regex=True)

In [21]:
# Remove Delhi District from census names and readd as Delhi
census_names = census_names[~((census_names['State'] == 'Delhi District'))]

delhi_districts = pd.DataFrame({
    'State': ['Delhi'],
    'District': ['Delhi']
})
census_names = pd.concat([census_names, delhi_districts], ignore_index=True)

In [22]:
#Add alias column and change names for Tamil Nadu
census_names['Alias'] = ''
census_names.loc[(census_names['State'] == 'Tamil Nadu') & (census_names['District'] == 'Chengai-Anna'), 
                 'Alias'] = 'Chengai-Anna'
census_names.loc[(census_names['State'] == 'Tamil Nadu') & (census_names['District'] == 'Dindigul-Quaid-E-Milleth'), 
                 'Alias'] = ['Dindigul-Quaid-E-Milleth']

#Change names to more common spellings
census_names.loc[(census_names['State'] == 'Tamil Nadu') & (census_names['District'] == 'Chengai-Anna'), 
                 'District'] = 'Chengalpattu'
census_names.loc[(census_names['State'] == 'Tamil Nadu') & (census_names['District'] == 'Dindigul-Quaid-E-Milleth'), 
                 'District'] = 'Dindigul Anna'

# Remove hyphen in North Arcot-Ambedker
census_names.loc[(census_names['State'] == 'Tamil Nadu') & (census_names['District'] == 'North Arcot-Ambedker'), 
                 'District'] = 'North Arcot Ambedkar'

In [23]:
#Change spelling for three districts in Uttar Pradesh
census_names.loc[(census_names['State'] == 'Uttar Pradesh') & (census_names['District'] == 'Kanpur (dehat)'), 
                 'District'] = 'Kanpur Dehat'
census_names.loc[(census_names['State'] == 'Uttar Pradesh') & (census_names['District'] == 'Kanpur (nagar)'), 
                 'District'] = 'Kanpur Nagar'

#Change spelling for Uttarkashi and add to alias column
census_names.loc[(census_names['State'] == 'Uttar Pradesh') & (census_names['District'] == 'Uttarkashi'), 
                 'District'] = 'Uttar Kashi'
census_names.loc[(census_names['State'] == 'Uttar Pradesh') & (census_names['District'] == 'Uttar Kashi'), 
                 'Alias'] = 'Uttarkashi'


In [24]:
#Sort and inspect final census names
census_names.sort_values(by=['State', 'District'], inplace=True)
print(census_names)

                           State                    District Alias
0    Andaman And Nicobar Islands                    Andamans      
1    Andaman And Nicobar Islands                    Nicobars      
2                 Andhra Pradesh                    Adilabad      
3                 Andhra Pradesh                   Anantapur      
4                 Andhra Pradesh                    Chittoor      
..                           ...                         ...   ...
435                  West Bengal                       Nadia      
436                  West Bengal  North Twenty Four Parganas      
437                  West Bengal                    Puruliya      
438                  West Bengal  South Twenty Four Parganas      
439                  West Bengal               West Dinajpur      

[466 rows x 3 columns]


In [25]:
census_names.to_csv('census_1991_districts.csv', index=False)